# 🎨 Kaggle All-in-One AI Studio (2x Tesla T4 16GB)
### Chuẩn OpenAI REST API trực tiếp qua Cloudflare Public Tunnel:
- 👁️ **VLM**: Qwen3.8 27B GGUF (UD-Q4_K_M + Vision mmproj trên Dual-GPU)
- 🎙️ **STT**: Whisper-large-v3-turbo (**Bản FULL FP16** trên GPU 0)
- 🔊 **TTS**: Kokoro-82M (**Bản FULL FP16** trên GPU 0)
- 🖼️ **GenImage**: FLUX.1-schnell (4-bit NF4 trên GPU 1)
- 🎬 **GenVideo**: Wan2.1 / Wan2.2 (Text-to-Video trên GPU 1)
- 🌐 **Endpoint**: Xuất trực tiếp URL Public chuẩn OpenAI ()

In [ ]:
# 1. Nạp biến môi trường tự động (Hỗ trợ Kaggle Secrets, file .env, hoặc cấu hình runtime)
import os, json

# Thiết lập tối ưu GPU & Cache
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["IDLE_SHUTDOWN_MINUTES"] = "120"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/tmp/hf_cache"

# Nạp Kaggle UserSecrets nếu được cấp quyền
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN", "NTFY_TOPIC"]:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

# Đọc file .env nếu có sẵn
for env_candidate in [".env", "/kaggle/working/Gen_Image-Video/kaggle/all-in-one/.env"]:
    if os.path.exists(env_candidate):
        try:
            with open(env_candidate, "r") as ef:
                for line in ef:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        clean_val = v.strip().strip("\"").strip("\x27")
                        os.environ.setdefault(k.strip(), clean_val)
        except Exception:
            pass

# Tạo cấu hình ~/.kaggle/kaggle.json để Kaggle API / CLI hoạt động thông suốt
try:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as kf:
            json.dump({
                "username": os.environ["KAGGLE_USERNAME"],
                "key": os.environ["KAGGLE_KEY"]
            }, kf)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

print(f"✅ Môi trường cấu hình hoàn tất! User: {os.environ.get('KAGGLE_USERNAME', 'Chưa cấu hình')}")


In [ ]:
# 2. Đồng bộ mã nguồn Studio AI (Live Payload)
import os, base64, io, tarfile

target_dir = "/kaggle/working/Gen_Image-Video/kaggle/all-in-one"
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)

payload_data = "..."
raw_tar = base64.b64decode(payload_data)
with tarfile.open(fileobj=io.BytesIO(raw_tar), mode="r:gz") as tar:
    tar.extractall(".")

print("✅ Toàn bộ module Studio AI phiên bản 3.5 đã đồng bộ thành công!")


In [ ]:
# --- CELL 3: SETUP ENVIRONMENT (100% Offline via kaggle-vlm-wheels + Fallback) ---
import glob, os, shutil, subprocess, sys, time, urllib.request

def _ntfy_dbg(msg: str):
    try:
        u = os.environ.get("KAGGLE_USERNAME", "cuongnguyen1802")
        req = urllib.request.Request(f"https://ntfy.sh/studio-ai-debug-{u}", data=msg.encode("utf-8"), method="POST")
        urllib.request.urlopen(req, timeout=4)
    except Exception:
        pass

t_setup_0 = time.time()
_ntfy_dbg("[VLM] Cell 3 starting environment setup...")
bundles = glob.glob("/kaggle/input/**/vlm_runtime_bundle.tar.bin", recursive=True)

if bundles:
    bundle_path = bundles[0]
    print(f"🚀 [Offline Mode] Đang giải nén llama-server CUDA 12.8 & Noble runtime từ: {bundle_path}")
    subprocess.run(["tar", "-xzf", bundle_path, "-C", "/tmp"], check=True)

    print("📦 [1/2] Cài đặt offline cloudflared & FastAPI wheels...")
    subprocess.run("dpkg -i --force-depends /tmp/debs/*.deb >/dev/null 2>&1", shell=True, check=False)
    for whl in sorted(glob.glob("/tmp/wheels/*.whl")):
        subprocess.run(
            f"{sys.executable} -m pip install -q --no-index --no-deps {whl}",
            shell=True,
            check=False,
        )
    shutil.rmtree("/tmp/debs", ignore_errors=True)
    shutil.rmtree("/tmp/wheels", ignore_errors=True)

# Đảm bảo đủ toàn bộ sub-dependencies của FastAPI / Uvicorn (starlette, annotated-doc, python-multipart...)
try:
    import fastapi, uvicorn, starlette, yaml, multipart
except Exception as imp_err:
    print(f"📦 Bổ sung nhanh các gói phụ trợ FastAPI ({imp_err})...")
    _ntfy_dbg(f"[VLM] Installing missing FastAPI deps ({imp_err})...")
    subprocess.run(
        f"{sys.executable} -m pip install -q fastapi uvicorn pyyaml python-multipart requests pillow psutil",
        shell=True,
        check=False,
    )

if not glob.glob("/tmp/llama-cpp/**/llama-server", recursive=True):
    print("⚠️ Không tìm thấy llama-server trong bundle, tải online...")
    _ntfy_dbg("[VLM] Downloading llama-server online fallback...")
    os.makedirs("/tmp/llama-cpp", exist_ok=True)
    os.makedirs("/tmp/noble-libs", exist_ok=True)
    setup_cmd = (
        "(curl -sL --retry 3 https://github.com/ggml-org/llama.cpp/releases/download/b11184/llama-b11184-bin-ubuntu-cuda-12.8-x64.tar.gz | tar -xz -C /tmp/llama-cpp) & "
        "(curl -sL --retry 3 https://github.com/ggml-org/llama.cpp/releases/download/b11184/cudart-llama-b11184-bin-ubuntu-cuda-12.8-x64.tar.gz | tar -xz -C /tmp/llama-cpp) & "
        "(curl -sL --retry 3 -o /tmp/libc6.deb http://archive.ubuntu.com/ubuntu/pool/main/g/glibc/libc6_2.39-0ubuntu8_amd64.deb && dpkg-deb -x /tmp/libc6.deb /tmp/noble-libs) & "
        "(curl -sL --retry 3 -o /tmp/libstdcpp6.deb http://archive.ubuntu.com/ubuntu/pool/main/g/gcc-14/libstdc++6_14-20240412-0ubuntu1_amd64.deb && dpkg-deb -x /tmp/libstdcpp6.deb /tmp/noble-libs) & "
        "(curl -sL --retry 3 -o /tmp/libgcc.deb http://archive.ubuntu.com/ubuntu/pool/main/g/gcc-14/libgcc-s1_14-20240412-0ubuntu1_amd64.deb && dpkg-deb -x /tmp/libgcc.deb /tmp/noble-libs) & "
        "wait && chmod -R +x /tmp/llama-cpp"
    )
    subprocess.run(setup_cmd, shell=True)

found_bins = glob.glob("/tmp/llama-cpp/**/llama-server", recursive=True)
llama_bin = found_bins[0] if found_bins else "/tmp/llama-cpp/llama-b11184/llama-server"
bin_dir = os.path.dirname(os.path.abspath(llama_bin))
os.environ["LLAMA_SERVER_BIN"] = llama_bin
os.environ["GGML_BACKEND_DIR"] = bin_dir

try:
    noble_lds = glob.glob("/tmp/noble-libs/**/ld-linux-x86-64.so.2", recursive=True)
    noble_ld = os.path.realpath(noble_lds[0]) if noble_lds else "/tmp/noble-libs/usr/lib/x86_64-linux-gnu/ld-linux-x86-64.so.2"
    local_ld = os.path.join(bin_dir, "ld-linux-x86-64.so.2")
    if not os.path.exists(local_ld):
        shutil.copy2(noble_ld, local_ld)
        os.chmod(local_ld, 0o755)
    lib_path = f"{bin_dir}:/tmp/noble-libs/lib/x86_64-linux-gnu:/tmp/noble-libs/usr/lib/x86_64-linux-gnu:/usr/local/cuda/lib64:/usr/local/nvidia/lib64:/usr/lib/x86_64-linux-gnu:/lib/x86_64-linux-gnu"
    ver_check = subprocess.run([local_ld, "--library-path", lib_path, llama_bin, "--version"], cwd=bin_dir, capture_output=True, text=True)
    msg = f"[VLM] Cell 3 llama-server check (code={ver_check.returncode}) done in {time.time() - t_setup_0:.1f}s!"
    print("✅", msg)
    _ntfy_dbg(msg)
except Exception as e:
    print(f"⚠️ llama-server check warning: {e}")
    _ntfy_dbg(f"[VLM] Cell 3 llama-server warning: {e}")


In [ ]:
# 4. Khởi động Studio (all-in-one mode - Hỗ trợ Intentional Shutdown exit 99)
import os, glob, subprocess, sys, time
os.environ["STUDIO_MODE"] = "vlm"
found_bins = glob.glob("/tmp/llama-cpp/**/llama-server", recursive=True)
if found_bins:
    os.environ["LLAMA_SERVER_BIN"] = found_bins[0]
for attempt in range(3):
    t0 = time.time()
    ret = subprocess.run([sys.executable, "-u", "main.py", "--mode", "vlm"], cwd=os.getcwd())
    if ret.returncode in (0, 99):
        print(f"Clean exit requested ({ret.returncode}). Stopping notebook.")
        break
    if time.time() - t0 < 15:
        print(f"Fast crash (code {ret.returncode}). Exiting to preserve error log.")
        break
    print(f"Server exited with code {ret.returncode}, restarting in 3s...")
    time.sleep(3)
